# RLHF Preference Alignment: LoRA vs Custom DoRA on Anthropic HH-RLHF

This notebook is separate from the commonsense-reasoning notebook so your original experiment can keep its package pins:

```python
transformers==4.44.2
accelerate==0.34.2
datasets==2.21.0
```

It compares **manual DPO + PEFT LoRA** against **manual DPO + your custom DoRA implementation**. It does not import TRL, which avoids the TRL/Transformers version conflict.

In [ ]:
# Keep this compatible with the initial commonsense setup. No TRL is used here.
%pip install -q --no-cache-dir --upgrade-strategy only-if-needed \
    "transformers==4.44.2" \
    "accelerate==0.34.2" \
    "datasets==2.21.0" \
    "peft==0.12.0" \
    "sentencepiece" \
    "safetensors" \
    "tqdm"

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import time
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

import transformers
import accelerate
import datasets
import peft
from datasets import load_dataset, Dataset as HFDataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType

print('transformers', transformers.__version__)
print('accelerate', accelerate.__version__)
print('datasets', datasets.__version__)
print('peft', peft.__version__)
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
# Optional: run this if BASE_MODEL is gated.
# from huggingface_hub import login
# login()
from huggingface_hub import login, whoami

login()
print(whoami())

In [ ]:
# -----------------------------
# Experiment config
# -----------------------------
DORA_ROOT = Path('/content/DoRA')
if not DORA_ROOT.exists():
    DORA_ROOT = Path('/content/drive/MyDrive/DoRA')
DORA_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT = DORA_ROOT / 'outputs_hh_rlhf_manual_dpo'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Use a model you have access to.
# BASE_MODEL = 'yahma/llama-13b-hf'
# BASE_MODEL = 'meta-llama/Llama-2-7b-hf'
# BASE_MODEL = 'meta-llama/Meta-Llama-3-8B'
BASE_MODEL = "yahma/llama-7b-hf"


TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'up_proj', 'down_proj']
DPO_METHODS = ['LoRA', 'DoRA']
DPO_RANKS = [16, 32]
DROPOUT = 0.05

# Start small, then scale up.
DPO_MAX_TRAIN_SAMPLES = 5000
DPO_MAX_EVAL_SAMPLES = 1000
DPO_EPOCHS = 1
DPO_BETA = 0.1
DPO_LEARNING_RATE = 2e-4
DPO_WARMUP_STEPS = 20
DPO_MAX_LENGTH = 512
DPO_MICRO_BATCH_SIZE = 1
DPO_GRADIENT_ACCUMULATION_STEPS = 8
DPO_MAX_GRAD_NORM = 1.0
DPO_LOGGING_STEPS = 10

print('BASE_MODEL:', BASE_MODEL)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('DPO_METHODS:', DPO_METHODS)
print('DPO_RANKS:', DPO_RANKS)

In [ ]:
def load_tokenizer(base_model: str):
    tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    return tokenizer


tokenizer = load_tokenizer(BASE_MODEL)
print('pad token:', tokenizer.pad_token, tokenizer.pad_token_id)

In [ ]:
def split_hh_pair(chosen_text: str, rejected_text: str) -> Optional[Dict[str, str]]:
    """Split Anthropic HH-RLHF chosen/rejected transcripts into prompt and assistant replies.

    Some HH rows have awkward common-prefix boundaries. Return None for rows that
    cannot be cleanly split so the loader can skip them instead of crashing.
    """
    chosen_text = str(chosen_text)
    rejected_text = str(rejected_text)

    shared_len = 0
    for a, b in zip(chosen_text, rejected_text):
        if a != b:
            break
        shared_len += 1
    shared = chosen_text[:shared_len]

    markers = ['\n\nAssistant:', 'Assistant:']
    boundary = -1
    marker_used = ''
    for marker in markers:
        boundary = shared.rfind(marker)
        if boundary >= 0:
            marker_used = marker
            break

    if boundary >= 0:
        prompt_end = boundary + len(marker_used)
    else:
        # Fallback: use the last complete line in the common prefix.
        prompt_end = shared.rfind('\n')
        if prompt_end < 0:
            return None

    prompt = chosen_text[:prompt_end].rstrip() + ' '
    chosen = chosen_text[prompt_end:].strip()
    rejected = rejected_text[prompt_end:].strip()

    if not prompt.strip() or not chosen.strip() or not rejected.strip() or chosen == rejected:
        return None
    return {'prompt': prompt, 'chosen': chosen, 'rejected': rejected}


def load_hh_rlhf_splits() -> Tuple[HFDataset, HFDataset]:
    raw = load_dataset('Anthropic/hh-rlhf')

    def convert(split_name: str, max_samples: Optional[int]) -> HFDataset:
        split = raw[split_name].shuffle(seed=SEED)
        rows = []
        skipped = 0
        target = max_samples if max_samples is not None else len(split)

        for ex in split:
            row = split_hh_pair(ex['chosen'], ex['rejected'])
            if row is None:
                skipped += 1
                continue

            rows.append(row)

            if len(rows) >= target:
                break

        print(f'{split_name}: kept {len(rows)}, skipped {skipped}')
        return HFDataset.from_list(rows)

    train_ds = convert('train', DPO_MAX_TRAIN_SAMPLES)
    eval_ds = convert('test', DPO_MAX_EVAL_SAMPLES)

    train_path = OUTPUT_ROOT / f'hh_rlhf_train_seed{SEED}_n{len(train_ds)}.json'
    eval_path = OUTPUT_ROOT / f'hh_rlhf_eval_seed{SEED}_n{len(eval_ds)}.json'

    train_path.write_text(json.dumps(train_ds.to_list(), indent=2), encoding='utf-8')
    eval_path.write_text(json.dumps(eval_ds.to_list(), indent=2), encoding='utf-8')

    print('Saved train manifest:', train_path)
    print('Saved eval manifest:', eval_path)

    print('\nFirst 10 training samples:')
    for i, ex in enumerate(train_ds.select(range(min(10, len(train_ds))))):
        print('\n' + '=' * 80)
        print(f'TRAIN SAMPLE {i + 1}')
        print('-' * 80)
        print('PROMPT:')
        print(ex['prompt'][:1000])
        print('\nCHOSEN:')
        print(ex['chosen'][:1000])
        print('\nREJECTED:')
        print(ex['rejected'][:1000])

    return train_ds, eval_ds


train_ds, eval_ds = load_hh_rlhf_splits()

In [ ]:
class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float):
        super().__init__()
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        self.weight = nn.Parameter(base.weight.detach().clone(), requires_grad=False)
        self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False) if base.bias is not None else None

        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features, device=self.weight.device, dtype=self.weight.dtype))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=self.weight.device, dtype=self.weight.dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        magnitude = torch.linalg.vector_norm(self.weight.float(), ord=2, dim=1).to(device=self.weight.device, dtype=self.weight.dtype)
        self.magnitude = nn.Parameter(magnitude)

    def merged_weight(self):
        directional = self.weight + self.scaling * (self.lora_B @ self.lora_A)
        norm = torch.linalg.vector_norm(directional.float(), ord=2, dim=1, keepdim=True).to(directional.dtype).clamp_min(1e-6).detach()
        return self.magnitude[:, None] * directional / norm

    def forward(self, x):
        if self.training:
            directional = self.weight + self.scaling * (self.lora_B @ self.lora_A)
            norm = torch.linalg.vector_norm(directional.float(), ord=2, dim=1).to(directional.dtype).clamp_min(1e-6).detach()
            norm_scale = self.magnitude / norm
            base_out = F.linear(x, self.weight, self.bias)
            lora_out = F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B) * self.scaling
            return norm_scale * (base_out + lora_out)
        return F.linear(self.dropout(x), self.merged_weight(), self.bias)


def replace_target_linears_with_dora(model: nn.Module, target_modules: Iterable[str], rank: int, alpha: int, dropout: float) -> int:
    modules = dict(model.named_modules())
    replacements = []
    target_modules = tuple(target_modules)
    for name, module in modules.items():
        if isinstance(module, nn.Linear) and any(target == name.split('.')[-1] or target in name for target in target_modules):
            parent_name, child_name = name.rsplit('.', 1) if '.' in name else ('', name)
            parent = modules[parent_name] if parent_name else model
            replacements.append((parent, child_name, name))
    for parent, child_name, full_name in replacements:
        setattr(parent, child_name, DoRALinear(getattr(parent, child_name), rank=rank, alpha=alpha, dropout=dropout))
    return len(replacements)


def freeze_all_then_unfreeze_dora(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    for module in model.modules():
        if isinstance(module, DoRALinear):
            module.lora_A.requires_grad = True
            module.lora_B.requires_grad = True
            module.magnitude.requires_grad = True


def trainable_parameter_summary(model: nn.Module) -> Tuple[int, int, float]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total, 100.0 * trainable / total


def save_dora_adapter(model: nn.Module, path: Path, rank: int):
    path.parent.mkdir(parents=True, exist_ok=True)
    state = {}
    for name, module in model.named_modules():
        if isinstance(module, DoRALinear):
            state[name + '.lora_A'] = module.lora_A.detach().cpu()
            state[name + '.lora_B'] = module.lora_B.detach().cpu()
            state[name + '.magnitude'] = module.magnitude.detach().cpu()
    torch.save({'rank': rank, 'alpha': 2 * rank, 'target_modules': TARGET_MODULES, 'state_dict': state}, path)
    print('Saved custom DoRA adapter:', path)

In [ ]:
def load_base_model(trainable: bool = False):
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=dtype,
        device_map='auto',
        low_cpu_mem_usage=True,
    )
    if hasattr(model, 'config'):
        model.config.use_cache = False
    if trainable:
        model.gradient_checkpointing_enable()
        if hasattr(model, 'enable_input_require_grads'):
            model.enable_input_require_grads()
    else:
        model.eval()
        for p in model.parameters():
            p.requires_grad = False
    return model


def load_lora_policy(rank: int):
    model = load_base_model(trainable=True)
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=rank,
        lora_alpha=2 * rank,
        lora_dropout=DROPOUT,
        bias='none',
        target_modules=TARGET_MODULES,
    )
    model = get_peft_model(model, config)
    model.print_trainable_parameters()
    return model


def load_dora_policy(rank: int):
    model = load_base_model(trainable=True)
    replaced = replace_target_linears_with_dora(model, TARGET_MODULES, rank=rank, alpha=2 * rank, dropout=DROPOUT)
    freeze_all_then_unfreeze_dora(model)
    trainable, total, pct = trainable_parameter_summary(model)
    print(f'Custom DoRA rank {rank}: replaced {replaced} linear layers')
    print(f'Custom DoRA rank {rank}: trainable {trainable:,} / {total:,} = {pct:.2f}%')
    return model

In [ ]:
def tokenize_prompt_completion_batch(examples: Sequence[dict], key: str, tokenizer, max_length: int):
    texts = [ex['prompt'] + ex[key] for ex in examples]
    prompt_ids = [tokenizer(ex['prompt'], add_special_tokens=False).input_ids for ex in examples]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt', add_special_tokens=False)
    labels = enc.input_ids.clone()
    for i, pids in enumerate(prompt_ids):
        mask_len = min(len(pids), labels.shape[1])
        labels[i, :mask_len] = -100
    labels[enc.attention_mask == 0] = -100
    return enc.input_ids, enc.attention_mask, labels


def dpo_collate(batch: Sequence[dict]):
    chosen_ids, chosen_mask, chosen_labels = tokenize_prompt_completion_batch(batch, 'chosen', tokenizer, DPO_MAX_LENGTH)
    rejected_ids, rejected_mask, rejected_labels = tokenize_prompt_completion_batch(batch, 'rejected', tokenizer, DPO_MAX_LENGTH)
    out = {
        'chosen_input_ids': chosen_ids,
        'chosen_attention_mask': chosen_mask,
        'chosen_labels': chosen_labels,
        'rejected_input_ids': rejected_ids,
        'rejected_attention_mask': rejected_mask,
        'rejected_labels': rejected_labels,
    }
    if 'ref_chosen_logp' in batch[0]:
        out['ref_chosen_logp'] = torch.tensor([ex['ref_chosen_logp'] for ex in batch], dtype=torch.float32)
        out['ref_rejected_logp'] = torch.tensor([ex['ref_rejected_logp'] for ex in batch], dtype=torch.float32)
    return out


def sequence_logps(model, input_ids, attention_mask, labels):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    shifted_labels = labels[:, 1:]
    loss_mask = shifted_labels != -100
    safe_labels = shifted_labels.masked_fill(~loss_mask, 0)
    log_probs = F.log_softmax(logits.float(), dim=-1)
    token_logps = torch.gather(log_probs, dim=-1, index=safe_labels.unsqueeze(-1)).squeeze(-1)
    return (token_logps * loss_mask).sum(dim=-1)


def move_batch_to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}

In [ ]:
def add_reference_logps(dataset: HFDataset, split_name: str) -> HFDataset:
    cache_path = OUTPUT_ROOT / f'ref_logps_{split_name}_{BASE_MODEL.replace("/", "_")}_n{len(dataset)}.json'
    if cache_path.exists():
        print('Loading cached ref logps:', cache_path)
        return HFDataset.from_list(json.loads(cache_path.read_text()))

    ref_model = load_base_model(trainable=False)
    raw_loader = DataLoader(dataset.to_list(), batch_size=DPO_MICRO_BATCH_SIZE, shuffle=False, collate_fn=lambda x: x)
    tensor_loader = DataLoader(dataset.to_list(), batch_size=DPO_MICRO_BATCH_SIZE, shuffle=False, collate_fn=dpo_collate)
    rows = []
    for raw_batch, tensor_batch in tqdm(zip(raw_loader, tensor_loader), total=len(tensor_loader), desc=f'ref logps {split_name}'):
        tensor_batch = move_batch_to_device(tensor_batch, ref_model.device)
        with torch.no_grad():
            chosen = sequence_logps(ref_model, tensor_batch['chosen_input_ids'], tensor_batch['chosen_attention_mask'], tensor_batch['chosen_labels'])
            rejected = sequence_logps(ref_model, tensor_batch['rejected_input_ids'], tensor_batch['rejected_attention_mask'], tensor_batch['rejected_labels'])
        for ex, c, r in zip(raw_batch, chosen.detach().cpu().tolist(), rejected.detach().cpu().tolist()):
            ex = dict(ex)
            ex['ref_chosen_logp'] = float(c)
            ex['ref_rejected_logp'] = float(r)
            rows.append(ex)

    del ref_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    cache_path.write_text(json.dumps(rows, indent=2), encoding='utf-8')
    print('Saved ref logps:', cache_path)
    return HFDataset.from_list(rows)


train_ds = add_reference_logps(train_ds, 'train')
eval_ds = add_reference_logps(eval_ds, 'eval')

In [ ]:
def dpo_loss(policy_chosen_logp, policy_rejected_logp, ref_chosen_logp, ref_rejected_logp, beta: float):
    pi_logratio = policy_chosen_logp - policy_rejected_logp
    ref_logratio = ref_chosen_logp - ref_rejected_logp
    logits = pi_logratio - ref_logratio
    losses = -F.logsigmoid(beta * logits)
    rewards_chosen = beta * (policy_chosen_logp - ref_chosen_logp)
    rewards_rejected = beta * (policy_rejected_logp - ref_rejected_logp)
    return losses.mean(), rewards_chosen.detach(), rewards_rejected.detach()


def train_dpo_policy(method: str, rank: int):
    if method == 'LoRA':
        model = load_lora_policy(rank)
    elif method == 'DoRA':
        model = load_dora_policy(rank)
    else:
        raise ValueError(method)

    train_loader = DataLoader(train_ds.to_list(), batch_size=DPO_MICRO_BATCH_SIZE, shuffle=True, collate_fn=dpo_collate)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=DPO_LEARNING_RATE)
    update_steps_per_epoch = math.ceil(len(train_loader) / DPO_GRADIENT_ACCUMULATION_STEPS)
    total_steps = DPO_EPOCHS * update_steps_per_epoch
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=DPO_WARMUP_STEPS, num_training_steps=total_steps)

    model.train()
    optimizer.zero_grad(set_to_none=True)
    global_step = 0
    running = 0.0
    start_time = time.time()

    for epoch in range(DPO_EPOCHS):
        pbar = tqdm(train_loader, desc=f'DPO {method} r{rank} epoch {epoch + 1}/{DPO_EPOCHS}')
        for step, batch in enumerate(pbar):
            batch = move_batch_to_device(batch, model.device)
            policy_chosen = sequence_logps(model, batch['chosen_input_ids'], batch['chosen_attention_mask'], batch['chosen_labels'])
            policy_rejected = sequence_logps(model, batch['rejected_input_ids'], batch['rejected_attention_mask'], batch['rejected_labels'])
            loss, chosen_rewards, rejected_rewards = dpo_loss(
                policy_chosen,
                policy_rejected,
                batch['ref_chosen_logp'],
                batch['ref_rejected_logp'],
                DPO_BETA,
            )
            (loss / DPO_GRADIENT_ACCUMULATION_STEPS).backward()
            running += float(loss.detach().cpu())

            if (step + 1) % DPO_GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), DPO_MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                if global_step % DPO_LOGGING_STEPS == 0:
                    margin = float((chosen_rewards - rejected_rewards).mean().cpu())
                    pbar.set_postfix({'loss': running / DPO_LOGGING_STEPS, 'reward_margin': margin, 'lr': scheduler.get_last_lr()[0]})
                    running = 0.0

    print(f'DPO {method} rank {rank} finished in {(time.time() - start_time) / 60:.1f} min, updates={global_step}')
    out_dir = OUTPUT_ROOT / f'{method}_rank{rank}'
    out_dir.mkdir(parents=True, exist_ok=True)
    if method == 'LoRA':
        model.save_pretrained(out_dir / 'adapter')
    else:
        save_dora_adapter(model, out_dir / f'custom_dora_rank{rank}.pt', rank)
    return model

In [ ]:
def evaluate_preference_accuracy(model, method: str, rank: int):
    """Evaluate both raw log-prob accuracy and DPO reward accuracy.

    Raw log-prob accuracy asks whether policy_logp(chosen) > policy_logp(rejected).
    This is length-biased and is not the DPO objective.

    DPO reward accuracy asks whether
        beta * (policy_logp(chosen) - ref_logp(chosen))
    is greater than
        beta * (policy_logp(rejected) - ref_logp(rejected)).
    This matches the implicit reward comparison optimized by DPO.
    """
    rows = []
    raw_correct = 0
    reward_correct = 0
    raw_loader = DataLoader(eval_ds.to_list(), batch_size=DPO_MICRO_BATCH_SIZE, shuffle=False, collate_fn=lambda x: x)
    tensor_loader = DataLoader(eval_ds.to_list(), batch_size=DPO_MICRO_BATCH_SIZE, shuffle=False, collate_fn=dpo_collate)
    model.eval()
    for raw_batch, batch in tqdm(zip(raw_loader, tensor_loader), total=len(tensor_loader), desc=f'eval {method} r{rank}'):
        batch = move_batch_to_device(batch, model.device)
        with torch.no_grad():
            policy_chosen = sequence_logps(model, batch['chosen_input_ids'], batch['chosen_attention_mask'], batch['chosen_labels'])
            policy_rejected = sequence_logps(model, batch['rejected_input_ids'], batch['rejected_attention_mask'], batch['rejected_labels'])

        ref_chosen = batch['ref_chosen_logp']
        ref_rejected = batch['ref_rejected_logp']
        chosen_reward = DPO_BETA * (policy_chosen - ref_chosen)
        rejected_reward = DPO_BETA * (policy_rejected - ref_rejected)

        for ex, pc, pr, rc, rr, cr, rj in zip(
            raw_batch,
            policy_chosen.cpu().tolist(),
            policy_rejected.cpu().tolist(),
            ref_chosen.cpu().tolist(),
            ref_rejected.cpu().tolist(),
            chosen_reward.cpu().tolist(),
            rejected_reward.cpu().tolist(),
        ):
            raw_ok = pc > pr
            reward_ok = cr > rj
            raw_correct += int(raw_ok)
            reward_correct += int(reward_ok)
            rows.append({
                'method': method,
                'rank': rank,
                'policy_chosen_logp': pc,
                'policy_rejected_logp': pr,
                'raw_logp_margin': pc - pr,
                'raw_logp_correct': raw_ok,
                'ref_chosen_logp': rc,
                'ref_rejected_logp': rr,
                'chosen_reward': cr,
                'rejected_reward': rj,
                'reward_margin': cr - rj,
                'reward_correct': reward_ok,
                'prompt': ex['prompt'],
                'chosen': ex['chosen'],
                'rejected': ex['rejected'],
            })
    raw_acc = 100.0 * raw_correct / len(rows)
    reward_acc = 100.0 * reward_correct / len(rows)
    mean_raw_margin = float(np.mean([r['raw_logp_margin'] for r in rows]))
    mean_reward_margin = float(np.mean([r['reward_margin'] for r in rows]))
    out_path = OUTPUT_ROOT / f'preference_eval_{method}_rank{rank}.csv'
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(
        f'{method} rank {rank}: raw logp acc={raw_acc:.1f}, '
        f'DPO reward acc={reward_acc:.1f}, '
        f'raw margin={mean_raw_margin:.4f}, reward margin={mean_reward_margin:.4f}'
    )
    return {
        'Method': method,
        'rank r': rank,
        'Raw LogP Acc.': round(raw_acc, 1),
        'DPO Reward Acc.': round(reward_acc, 1),
        'Raw LogP Margin': round(mean_raw_margin, 4),
        'DPO Reward Margin': round(mean_reward_margin, 4),
    }

In [ ]:
results = []

for method in DPO_METHODS:
    for rank in DPO_RANKS:
        print('\n' + '=' * 80)
        print(f'Running manual DPO + {method}, rank={rank}')
        policy = train_dpo_policy(method, rank)
        results.append(evaluate_preference_accuracy(policy, method, rank))
        del policy
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_path = OUTPUT_ROOT / 'manual_dpo_lora_vs_custom_dora_hh_rlhf_results.csv'
results_df.to_csv(results_path, index=False)
print('Saved:', results_path)
results_df